# Baseline Intent Classifier for Apple Support

This notebook establishes a baseline model using TF-IDF feature extraction paired with standard linear classifiers to predict intent from customer tweets.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Add project root to path
sys.path.append(os.path.abspath('..'))
from src.preprocess import clean_tweet


## 1. Load Data
Let's load our hand-labelled golden set of 200 customer support tweets.

In [2]:
df = pd.read_csv('../data/golden_set.csv')
print(f"Total examples: {len(df)}")
df['intent'].value_counts()

## 2. Text Preprocessing Exploration

In [3]:
sample_tweet = df['text'].iloc[0]
print("Raw:    ", sample_tweet)
print("Cleaned:", clean_tweet(sample_tweet))

df['cleaned_text'] = df['text'].apply(clean_tweet)
df[['text', 'cleaned_text', 'intent']].head(3)

## 3. Baseline Pipeline: TF-IDF + Logistic Regression

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['intent'], test_size=0.25, random_state=42, stratify=df['intent']
)

baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=2500)),
    ('clf', LogisticRegression(random_state=42))
])

baseline_pipeline.fit(X_train, y_train)
y_pred = baseline_pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))

## 4. Baseline Error Analysis

In [5]:
errors = X_test[y_test != y_pred]
print(f"Total misclassifications in test split: {len(errors)}")
for idx, true_lbl, pred_lbl in zip(errors.index, y_test[y_test != y_pred], y_pred[y_test != y_pred]):
    print(f"Tweet: {errors[idx]}")
    print(f"True: {true_lbl} | Pred: {pred_lbl}\n")